# 🔄 Comparação Final — Confiabilidade Estatística vs Machine Learning

## Contexto

Este projeto percorreu duas abordagens distintas para o mesmo problema de fundo: entender e prever a degradação de motores turbofan. Cada uma respondeu uma pergunta diferente, em uma escala diferente.

A **Análise de Sobrevivência** (notebook 02) caracterizou estatisticamente o comportamento da frota como um todo. Kaplan-Meier, Nelson-Aalen e o ajuste de distribuições paramétricas revelaram quando, em média, os motores tendem a falhar e com que padrão de risco. É uma visão populacional, não diz nada sobre um motor específico, mas diz muito sobre o comportamento agregado de 100 motores.

Os **modelos de Machine Learning** (notebooks 04 e 05) atacaram o problema de forma oposta, prever, para um motor individual, exatamente quantos ciclos de vida restam, com base no estado atual dos seus sensores. É uma visão pontual e operacional, sem qualquer pretensão de generalizar para a frota.

Essas duas abordagens não competem entre si. Elas respondem perguntas que a indústria precisa fazer em momentos diferentes da gestão de manutenção:

- *"Quantas peças de reposição devo ter em estoque para os próximos 6 meses?"* → pergunta de frota, resolvida por survival analysis
- *"Este motor específico, que está em operação agora, precisa de manutenção essa semana?"* → pergunta operacional, resolvida por machine learning

## O que este notebook faz

**1. Consolida os resultados preditivos** dos três modelos da série (Regressão Linear, XGBoost, LSTM) em uma tabela comparativa final.

**2. Visualiza, lado a lado, as duas perguntas centrais do projeto** — a curva de sobrevivência da frota e a precisão de predição individual dos melhores modelos.

**3. Discute, sob a ótica da manutenção industrial, quando cada abordagem deve ser usada** — não como exercício acadêmico, mas como guia prático de decisão.

**4. Encerra a série** com uma síntese dos principais achados ao longo dos seis notebooks.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'

def s_score(y_true, y_pred):
    d = y_pred - y_true
    return np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1))

In [ ]:
# Carregando os resultados de todos os notebooks anteriores

# Survival Analysis
durations = pd.read_csv('data/train_FD001.txt', sep='\\s+', header=None,
                         usecols=[0, 1], names=['unit_id', 'time_cycles']
                        ).groupby('unit_id')['time_cycles'].max()
event_observed = np.ones(len(durations))

kmf = KaplanMeierFitter()
kmf.fit(durations, event_observed=event_observed, label='Motores FD001')

# Modelos preditivos
y_pred_lstm = np.load('data/processed/y_pred_lstm.npy')
y_test_lstm = np.load('data/processed/y_test_seq_final.npy')

y_pred_xgb  = np.load('data/processed/y_pred_xgb.npy')
y_test_xgb  = np.load('data/processed/y_test_tab_final.npy')

print('Ground truth idêntico entre XGBoost e LSTM:', np.array_equal(y_test_xgb, y_test_lstm))

y_test = y_test_xgb

## 2. Visão de Frota — Análise de Sobrevivência

A curva de Kaplan-Meier construída no notebook 02 responde uma pergunta que nenhum modelo de machine learning deste projeto responde: qual a probabilidade de um motor, escolhido aleatoriamente da frota, ainda estar operacional após um certo número de ciclos.

Essa abordagem não usa nenhuma leitura de sensor. Ela parte apenas do histórico de tempo até a falha de cada motor, sem olhar para o estado interno do equipamento em nenhum momento da sua vida. É uma visão agregada, construída sobre 100 observações completas, e seu valor está justamente nessa agregação.

Na prática industrial, esse tipo de análise sustenta decisões de planejamento. Uma equipe de manutenção que precisa decidir quantas peças de reposição manter em estoque para os próximos seis meses, ou quando programar uma parada preventiva para um lote inteiro de equipamentos do mesmo modelo, está fazendo exatamente o tipo de pergunta que a curva de sobrevivência responde.

A mediana de 199 ciclos encontrada no notebook 02 não diz nada sobre o motor número 37 especificamente. Diz que, entre 100 motores nessa frota, metade já terá falhado antes desse ponto. É informação estatística sobre o grupo, não sobre o indivíduo.

![Kaplan-Meier — Curva de Sobrevivência](midia/kaplan_meier.png)

## 3. Visão Individual — Machine Learning

Os modelos de Regressão Linear, Random Forest, XGBoost e LSTM, desenvolvidos nos notebooks 04 e 05, atacam o problema de uma forma completamente diferente da análise de sobrevivência. Em vez de olhar para o histórico agregado de falhas, esses modelos recebem as leituras dos sensores de um motor específico, naquele momento, e estimam quantos ciclos de vida restam para aquele equipamento individual.

Essa é a pergunta que importa quando alguém precisa decidir se um motor específico, em operação agora, deve ser retirado de serviço nesta semana ou pode continuar rodando com segurança. Não interessa o comportamento médio da frota. Interessa o estado real daquele ativo.

Dos quatro modelos testados ao longo do projeto, dois se destacaram. O XGBoost, usando rolling features extraídas dos sensores, alcançou RMSE de 17.78 ciclos. O LSTM, processando diretamente a sequência bruta de 30 ciclos sem qualquer feature engineering explícita, reduziu esse erro para 13.52 ciclos.

A diferença entre os dois ilustra um ponto importante sobre dados de séries temporais industriais. O XGBoost depende de alguém decidir, de antemão, quais estatísticas resumem bem o comportamento temporal de um sensor (média móvel, desvio padrão móvel). O LSTM aprende essas representações por conta própria, diretamente da sequência, o que explica seu ganho de performance neste problema específico.

## 4. As Duas Perguntas, Lado a Lado

O gráfico abaixo coloca as duas abordagens uma ao lado da outra, exatamente para evidenciar que elas não competem entre si.

À esquerda, a curva de Kaplan-Meier mostra a probabilidade de sobrevivência de toda a frota ao longo dos ciclos. É uma resposta sobre o grupo, construída sem olhar para nenhum sensor, apenas para o histórico de quando cada motor falhou.

À direita, XGBoost e LSTM mostram suas predições individuais contra o valor real de RUL de cada um dos 100 motores de teste. É uma resposta sobre o indivíduo, construída a partir do estado atual dos sensores daquele motor específico.

Nenhum dos dois painéis substitui o outro. O painel esquerdo nunca vai dizer quanto tempo de vida resta no motor número 42 agora. O painel direito nunca vai dizer quantas peças de reposição uma planta inteira vai precisar no próximo semestre. São ferramentas para perguntas diferentes, e uma equipe de manutenção madura usa as duas, dependendo do tipo de decisão que precisa tomar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
fig.patch.set_facecolor('white')

# Paleta refinada
COLOR_SURVIVAL = '#2C5F8A'
COLOR_XGB      = '#D94F4F'
COLOR_LSTM     = '#2E86AB'
COLOR_IDEAL    = '#7A7A7A'

# ---------- Painel 1 — Visão de Frota ----------
ax = axes[0]
kmf.plot_survival_function(ax=ax, ci_show=True, color=COLOR_SURVIVAL, linewidth=2.5)
ax.fill_between(kmf.survival_function_.index,
                 kmf.confidence_interval_.iloc[:, 0],
                 kmf.confidence_interval_.iloc[:, 1],
                 color=COLOR_SURVIVAL, alpha=0.12, linewidth=0)
ax.axhline(0.5, color=COLOR_IDEAL, linestyle=(0, (5, 3)), linewidth=1.2, alpha=0.8)
ax.text(355, 0.52, 'S(t) = 0.50', fontsize=8.5, color=COLOR_IDEAL,
        ha='right', style='italic')

ax.set_title('Visão de Frota', fontweight='bold', fontsize=13, pad=14, color='#222')
ax.text(0.5, 1.015, '"Quando os motores começam a falhar?"',
        transform=ax.transAxes, ha='center', fontsize=10.5,
        style='italic', color='#555')
ax.set_xlabel('Ciclos de Operação', fontsize=10.5, color='#333')
ax.set_ylabel('Probabilidade de Sobrevivência', fontsize=10.5, color='#333')
ax.get_legend().remove()
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#999')
ax.tick_params(colors='#555', labelsize=9)
ax.grid(True, linestyle='--', alpha=0.25, color='#aaa')
ax.set_facecolor('#FAFAFA')

# ---------- Painel 2 — Visão Individual ----------
ax = axes[1]
ax.plot([0, 125], [0, 125], color=COLOR_IDEAL, linestyle=(0, (5, 3)),
        linewidth=1.3, alpha=0.8, zorder=1, label='Ideal')
ax.scatter(y_test_xgb, y_pred_xgb, alpha=0.55, color=COLOR_XGB, s=34,
           edgecolors='white', linewidth=0.4, label='XGBoost', zorder=2)
ax.scatter(y_test_lstm, y_pred_lstm, alpha=0.55, color=COLOR_LSTM, s=34,
           edgecolors='white', linewidth=0.4, label='LSTM', zorder=3)

ax.set_title('Visão Individual', fontweight='bold', fontsize=13, pad=14, color='#222')
ax.text(0.5, 1.015, '"Quanto resta neste motor, agora?"',
        transform=ax.transAxes, ha='center', fontsize=10.5,
        style='italic', color='#555')
ax.set_xlabel('RUL Real', fontsize=10.5, color='#333')
ax.set_ylabel('RUL Previsto', fontsize=10.5, color='#333')
ax.legend(fontsize=9.5, frameon=True, loc='upper left',
          facecolor='white', edgecolor='#ddd')
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#999')
ax.tick_params(colors='#555', labelsize=9)
ax.grid(True, linestyle='--', alpha=0.25, color='#aaa')
ax.set_facecolor('#FAFAFA')

fig.suptitle('Confiabilidade Estatística vs Machine Learning',
             fontsize=16, fontweight='bold', color='#1a1a1a', y=1.06)
fig.text(0.5, 0.985, 'Duas perguntas diferentes, sobre o mesmo problema de degradação',
         ha='center', fontsize=11, style='italic', color='#666')

plt.tight_layout()
plt.savefig('midia/comparacao_final.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

![Tabela Comparativa](midia/comparacao_final.png)

## 5. Discussão: Quando Usar Cada Abordagem

Este projeto testou quatro modelos preditivos ao longo da série, Regressão Linear, Random Forest, XGBoost e LSTM, mas a discussão a seguir foca no XGBoost e no LSTM. Os dois primeiros serviram como baseline e ficaram estatisticamente equivalentes entre si, sem agregar uma perspectiva adicional à comparação. O XGBoost foi o melhor representante da abordagem tabular, e é ele que se contrapõe ao LSTM nesta análise final, junto da análise de sobrevivência.

Os três modelos, análise de sobrevivência, XGBoost e LSTM, não competem entre si. Eles existem em planos diferentes de decisão dentro da gestão de manutenção, e a escolha de qual usar depende exclusivamente da pergunta que precisa ser respondida.

A análise de sobrevivência opera no nível da frota. Ela não exige nenhum sensor, apenas o histórico de tempo até falha de cada equipamento. Seu horizonte é de planejamento, meses ou trimestres à frente, e sua granularidade é agregada, ela informa sobre o comportamento esperado de um grupo de ativos, nunca sobre um equipamento específico. É a ferramenta certa para decidir orçamento de manutenção, política de estoque de peças de reposição ou intervalos de troca preventiva para um lote inteiro de máquinas do mesmo modelo.

XGBoost e LSTM operam no nível do ativo individual. Ambos exigem leitura contínua de sensores e, no caso do LSTM, uma janela de histórico recente do equipamento. Seu horizonte é operacional, dias ou semanas à frente, e sua granularidade é pontual, eles informam sobre o estado real de um motor específico, agora. São a ferramenta certa para decidir se um equipamento precisa ser retirado de operação esta semana, ou para priorizar qual entre vários ativos críticos merece inspeção primeiro.

A diferença entre os dois modelos de machine learning vai além da métrica de erro, e é aqui que a decisão prática de qual usar numa indústria se torna mais delicada.

O XGBoost depende de rolling features, estatísticas calculadas manualmente a partir da janela temporal dos sensores. Essa dependência de engenharia de atributos explícita tem um efeito colateral favorável, cada decisão do modelo pode ser rastreada até uma variável compreensível, como o feature importance e o SHAP exploraram no notebook 04. Um engenheiro de confiabilidade consegue abrir o modelo e justificar por que ele decidiu que um motor está perto da falha, apontando exatamente qual sensor, e em qual transformação, pesou mais na decisão. Essa rastreabilidade tem valor real numa indústria, onde decisões de parar um equipamento ou liberar uma frota para operação frequentemente precisam ser auditadas, defendidas para um gestor ou justificadas num relatório de qualidade.

O LSTM recebe a sequência bruta e aprende sozinho quais padrões temporais importam, sem que isso passe por uma etapa de engenharia explícita. Essa capacidade de aprender a dinâmica temporal por conta própria se traduziu em ganho real de precisão neste projeto, RMSE de 13.52 contra 17.78 do XGBoost, uma melhoria superior a 20 por cento. Mas o preço dessa precisão é a opacidade. Não existe um conjunto claro de pesos ou variáveis para apontar como justificativa de uma predição específica. Técnicas de interpretabilidade para redes neurais recorrentes existem, mas são mais custosas de implementar e menos imediatas de comunicar do que um gráfico de SHAP.

Esse trade-off entre precisão e interpretabilidade não tem resposta universal. Depende do contexto de risco da aplicação. Para um motor de aeronave, onde cada decisão de manutenção é regulada e auditada, a rastreabilidade do XGBoost pode pesar mais do que um ganho de poucos ciclos de precisão. Para um sistema de monitoramento de larga escala, onde milhares de previsões são geradas automaticamente e apenas os casos mais críticos são revisados por um humano, o ganho de precisão do LSTM pode justificar a perda de transparência. Existe também o custo de desenvolvimento e manutenção do modelo em si, treinar e ajustar uma rede neural recorrente exige mais tempo de engenharia, mais dados e mais cuidado com overfitting do que treinar um XGBoost, o que tem peso direto no custo de implantar e manter a solução ao longo do tempo.

Na prática industrial, as três abordagens convivem. Uma planta madura em gestão de confiabilidade usa a análise de sobrevivência para decisões estratégicas de médio prazo, e escolhe entre um modelo tabular ou sequencial para decisões operacionais de curto prazo, de acordo com o quanto a interpretabilidade pesa naquele contexto específico. Não é uma escolha entre estatística e machine learning, nem entre um modelo e outro de forma definitiva. É reconhecer que perguntas diferentes, e contextos de risco diferentes, exigem ferramentas diferentes.

## 6. Tabela Comparativa  — Modelos Preditivos

Para registro completo da série, a tabela abaixo reúne os três modelos preditivos testados ao longo do projeto, incluindo a Regressão Linear, que serviu como baseline.

In [ ]:
# Resultados consolidados de todos os notebooks da série
rmse_lr,  ss_lr  = 19.60, 906.49
rmse_xgb, ss_xgb = float(np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))), s_score(y_test_xgb, y_pred_xgb)
rmse_lstm, ss_lstm = float(np.sqrt(mean_squared_error(y_test_lstm, y_pred_lstm))), s_score(y_test_lstm, y_pred_lstm)

resultados_finais = pd.DataFrame({
    'Modelo':   ['Regressão Linear', 'XGBoost', 'LSTM'],
    'Abordagem': ['Tabular (baseline)', 'Tabular', 'Sequencial'],
    'RMSE':     [rmse_lr, rmse_xgb, rmse_lstm],
    'S-score':  [ss_lr,   ss_xgb,   ss_lstm]
}).sort_values('RMSE').reset_index(drop=True)

display(resultados_finais.style
    .background_gradient(subset=['RMSE'],    cmap='RdYlGn_r')
    .background_gradient(subset=['S-score'], cmap='RdYlGn_r')
    .format({'RMSE': '{:.2f}', 'S-score': '{:.2f}'})
)

## 7. Tabela Comparativa — Quando Usar Cada Abordagem

![Tabela Comparativa](midia/tabela_comparativa.png)

## 8. Conclusão Geral do Projeto

Este projeto percorreu seis notebooks com um único objetivo de fundo, entender a degradação de motores turbofan a ponto de poder antecipá-la, sob duas óticas complementares, a estatística clássica de confiabilidade e o machine learning preditivo.

A análise exploratória (notebook 01) revelou que apenas 12 dos 21 sensores carregam informação real de degradação, e que sensores térmicos como `temp_lpt_outlet` e `static_hpc_outlet` concentram o sinal mais forte. As métricas PHM, monotonicidade e prognostibilidade, deram a esse achado fundamentação na literatura de manutenção preditiva, não apenas na correlação observada.

A análise de sobrevivência (notebook 02) mostrou que os motores têm um período de vida mínimo garantido de 128 ciclos, e que a distribuição Log-Normal descreve melhor a vida útil da frota do que o Weibull, a distribuição padrão da literatura de confiabilidade. Esse resultado por si só já é um lembrete de que premissas da literatura precisam ser testadas, não assumidas, em cada novo conjunto de dados.

O feature engineering (notebook 03) e os modelos tabulares (notebook 04) demonstraram, na prática, que o preprocessing pode ter mais impacto no resultado final do que a escolha do algoritmo. A mesma Regressão Linear, com uma correção de normalização, teve seu RMSE reduzido de 58 para 19.6 ciclos, sem qualquer mudança no modelo.

O LSTM (notebook 05) superou os modelos tabulares ao aprender diretamente a dinâmica temporal dos sensores, sem depender de rolling features pré-calculadas, alcançando RMSE de 13.52 ciclos.

E este notebook final amarrou as duas abordagens estatísticas do projeto sob uma ótica que poucos materiais sobre CMAPSS exploram, reconhecendo que análise de sobrevivência e machine learning não competem entre si, mas respondem perguntas diferentes, em escalas diferentes, para decisões diferentes dentro da gestão de manutenção industrial.

O dataset CMAPSS é, na superfície, um benchmark acadêmico. Mas a forma como ele foi tratado aqui, com rigor estatístico, validação de premissas e conexão constante com a prática de confiabilidade industrial, é o que separa um exercício de modelagem de um projeto que efetivamente ensina como pensar sobre manutenção preditiva.